RAGAS Colab — End-to-end (AzureChatOpenAI)
A ready-to-run Colab-style notebook (Markdown + Python cells)

1. Installing and configuring Ragas with Azure OpenAI
2. Load configuration
3. Initialize Azure LLM wrapper for RAGAS
4. Create sample data (single-turn)
5. Run evaluation with built-in metrics
6. Example custom metric
Prerequisites for Colab
Google Colab runtime (Python 3.10+ recommended) An Azure OpenAI resource (endpoint, key, deployment name, model version) Set these as Colab environment variables or use os.environ in a cell.

Prerequisites for Local
Python 3.10+ recommended An Azure OpenAI resource (endpoint, key, deployment name, model version) VS Code and Running Chatbot.

1) Install dependencies

In [ ]:
%pip install \
    ragas==0.4.2 \
    langchain==0.3.27 \
    langchain-openai==0.3.28 \
    langchain-community==0.3.27 \
    langchain-huggingface==0.1.2 \
    openai==1.59.7 \
    datasets==3.2.0 \
    pandas==2.2.3 \
    sentence-transformers==3.4.1 \
    unstructured==0.16.11 \
    torch==2.5.1 \
    nest_asyncio==1.6.0
print('Skip pip install cell if packages are already installed')

In [ ]:
import configparser
import os
import pandas as pd
from datasets import Dataset
from ragas import SingleTurnSample, evaluate
from ragas.llms import LangchainLLMWrapper

print('Imports ready')


2. Configure Azure credentials (AzureChatOpenAI compatible)

In [ ]:
import os

# Set your Azure OpenAI credentials
os.environ["AZURE_OPENAI_API_KEY"] = "70jZZClor8BRjGKy3TrofnTMaYiNrY9St97Hk4aKJWbbLYRPB5gdJQQJ99BJACL93NaXJ3w3AAAAACOGbvnT"
os.environ["AZURE_OPENAI_API_VERSION"] = "2025-04-01-preview"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://azuse-mh1vouu8-australiaeast.openai.azure.com/"
os.environ["AZURE_DEPLOYMENT_NAME"] = "gpt-5-mini"

In [ ]:
api_key = os.environ.get('AZURE_OPENAI_API_KEY') or os.environ.get('OPENAI_API_KEY')
api_version = os.environ.get('AZURE_OPENAI_API_VERSION') or os.environ.get('OPENAI_API_VERSION')
endpoint = os.environ.get('AZURE_OPENAI_ENDPOINT') or os.environ.get('OPENAI_API_BASE')
deployment = os.environ.get('AZURE_DEPLOYMENT_NAME') or os.environ.get('DEPLOYMENT_NAME')

print('API key present?', bool(api_key))
print('Endpoint:', endpoint)
print('Deployment:', deployment)

Configuring AzureOpenAI for Evaluations.

In [ ]:
try:
    # Import inside try so the notebook cell remains robust if package names differ.
    # NOTE: AzureChatOpenAI now lives in langchain_openai (the langchain_community path is deprecated).
    from langchain_openai import AzureChatOpenAI
    from langchain_huggingface import HuggingFaceEmbeddings
    from ragas.embeddings import LangchainEmbeddingsWrapper
    use_langchain = True
except Exception as e:
    print('langchain_openai / langchain_huggingface not available. Error:', e)
    use_langchain = False

if use_langchain:
    # Create a LangChain AzureChatOpenAI instance using Azure-style parameters only
    llm_client = AzureChatOpenAI(
        azure_deployment=deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=1
    )
    wrapped_llm = LangchainLLMWrapper(llm_client, bypass_temperature=True)
    print('Wrapped LangChain AzureChatOpenAI for RAGAS')
else:
    # Fallback: user can manually create a wrapper around other Azure clients
    wrapped_llm = None
    print('No LLM wrapper created; set up your preferred Azure client and wrap it with LangchainLLMWrapper')

evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

3) Sample data
We transform your JSON-like example into a SingleTurnSample for RAGAS. Keep keys user_input, response, retrieved_contexts, reference / gold_answer.

In [ ]:
sample_case = {
    "input": "How much maternity leave are female employees entitled to?",
    "expected_output": "Female employees are entitled to 26 weeks of paid maternity leave as per the Maternity Benefit Act.",
    "actual_output": "Female employees receive 26 weeks of paid maternity leave, which can start up to 8 weeks before delivery.",
    "retrieval_context": [
        "Maternity leave provides 26 weeks of paid time off for female employees.",
        "Up to 8 weeks can be taken before the expected date of delivery."
    ],
    "actual_context": [
        "Female staff can take 26 weeks of paid maternity leave as per law.",
        "It ensures time for recovery and newborn care before and after birth."
    ]
}


mismatched_case = {
    "input": "What should employees wear on casual Fridays?",
    "expected_output": "Employees may wear clean jeans and simple t-shirts without rips or offensive prints.",
    "actual_output": "Employees can take one extra leave day on Fridays as part of work-life balance initiatives.",
    "retrieval_context": [
        "Casual Fridays allow jeans and casual shoes, provided clothing is clean and appropriate.",
        "Slogans or political prints are not permitted on t-shirts."
    ],
    "actual_context": [
        "Employees can avail earned leave for personal time or vacations.",
        "Leave requests must be approved through the HR portal."
    ]
}

In [ ]:
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import (
    AnswerCorrectness,
    AnswerSimilarity,
    Faithfulness,
    ContextRecall,
    ContextPrecision,
    NoiseSensitivity,
)
import pandas as pd
import asyncio

# Set up metric instances
answer_similarity = AnswerSimilarity(embeddings=evaluator_embeddings)

metrics = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    Faithfulness(llm=wrapped_llm),
    ContextRecall(llm=wrapped_llm),
    ContextPrecision(llm=wrapped_llm),
    NoiseSensitivity(llm=wrapped_llm),
]

sample1 = SingleTurnSample(
    user_input="What can employees wear on casual Fridays?",
    retrieved_contexts=["Clean jeans and plain t-shirts are allowed; no rips or offensive prints."],
    response="Employees may wear neat jeans and simple t-shirts on casual Fridays.",
    reference="Clean jeans and plain t-shirts without rips or prints."
)
sample2 = SingleTurnSample(
    user_input="How many sick leave days are allowed each year?",
    retrieved_contexts=["Employees are entitled to 7 days of sick leave annually with a medical note if absent over 3 days."],
    response="Employees get 7 sick leave days each year, with a doctor’s note if more than 3 days.",
    reference="7 days of sick leave per year."
)
sample3 = SingleTurnSample(
    user_input="Can employees book travel independently?",
    retrieved_contexts=["Travel must be booked through approved vendors unless written approval is given."],
    response="Employees can book travel only with prior written approval from the travel coordinator.",
    reference="Only with prior written approval from the travel coordinator or finance."
)

dataset = EvaluationDataset(samples=[sample1, sample2, sample3])

response = evaluate(dataset=dataset, metrics=metrics, llm=wrapped_llm, embeddings=evaluator_embeddings)
print(response.to_pandas().to_string())

In [ ]:
df = pd.DataFrame([{
    'user_input': sample_case['input'],
    'response': sample_case['actual_output'],
    'retrieved_contexts': sample_case['retrieval_context'],
    'reference': sample_case['expected_output']
}])
ragas_dataset = Dataset.from_pandas(df)
metric_instances = [
    AnswerCorrectness(llm=wrapped_llm, answer_similarity=answer_similarity),
    Faithfulness(llm=wrapped_llm),
    ContextRecall(llm=wrapped_llm)
]

try:
    # Reuse metric_instances from above if available
    eval_result = evaluate(dataset=dataset, metrics=metric_instances, llm=wrapped_llm, embeddings=evaluator_embeddings)
    print('Evaluate() returned:')
    print(eval_result)
except Exception as e:
    print('evaluate(...) failed:', e)
    print('Ensure ragas.evaluate signature and parameters match your installed version')

In [ ]:
# Required imports
from ragas.metrics import AspectCritic
from ragas.dataset_schema import SingleTurnSample
import nest_asyncio
import asyncio

nest_asyncio.apply()  # Enables async execution in Colab or notebooks

# Define your custom metrics
answer_correctness = AspectCritic(
    name="Answer Correctness",
    definition="Does the answer correctly convey the same meaning as the reference answer?",
    llm=wrapped_llm,
)

fluency = AspectCritic(
    name="Fluency",
    definition="Is the answer grammatically correct and easy to understand?",
    llm=wrapped_llm,
)

completeness = AspectCritic(
    name="Completeness",
    definition="Does the answer fully address the question using the provided context?",
    llm=wrapped_llm,
)

# Create a SingleTurnSample object from your mismatched_case dictionary
sample = SingleTurnSample(
    user_input=mismatched_case['input'],
    retrieved_contexts=mismatched_case['retrieval_context'],
    response=mismatched_case['actual_output'],
    reference=mismatched_case['expected_output']
)

# Async function to evaluate and print scores only
async def run_custom_metrics_scores_only():
    print("Sample:")
    print(f"  User Input: {sample.user_input}")
    print(f"  Response: {sample.response}")
    print(f"  Reference: {sample.reference}")
    print("\n  Metric Scores:")

    for metric in [answer_correctness, fluency, completeness]:
        score = await metric.single_turn_ascore(sample)
        print(f"    {metric.name}: {score}")

# Run the evaluation
await run_custom_metrics_scores_only()